In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Identificar la carpeta principal del proyecto
ruta_proyecto = Path.cwd()

if ruta_proyecto.name == "notebooks":
    ruta_proyecto = ruta_proyecto.parent

if str(ruta_proyecto) not in sys.path:
    sys.path.insert(0, str(ruta_proyecto))

print("Carpeta del proyecto:", ruta_proyecto)
print("Versión de pandas:", pd.__version__)

In [ ]:
from src.cargador_dataset import HFDatasetLoader

# Crear el cargador y obtener los datos de Hugging Face
cargador = HFDatasetLoader(
    dataset_id="SIH/palmer-penguins",
    split="train"
)

df = cargador.cargar_datos()

print("Dataset cargado correctamente.")
print("Cantidad de filas y columnas:", df.shape)

display(df.head())

In [ ]:
print("Nombres de las columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores faltantes por columna:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas:")
display(df.describe(include="all"))

In [ ]:
df_transformado = df.copy()

columnas_numericas = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

for columna in columnas_numericas:
    mediana = df_transformado[columna].median()
    df_transformado[columna] = (
        df_transformado[columna].fillna(mediana)
    )

df_transformado["sex"] = (
    df_transformado["sex"].fillna("unknown")
)

print("Valores faltantes después de la transformación:")
print(df_transformado.isnull().sum())

In [ ]:
df_transformado["bill_ratio"] = (
    df_transformado["bill_length_mm"]
    / df_transformado["bill_depth_mm"]
)

display(
    df_transformado[
        ["bill_length_mm", "bill_depth_mm", "bill_ratio"]
    ].head()
)

In [ ]:
df_transformado["weight_category"] = pd.cut(
    df_transformado["body_mass_g"],
    bins=[0, 3500, 4500, float("inf")],
    labels=["Ligero", "Medio", "Pesado"],
    include_lowest=True
)

print("Cantidad de pingüinos por categoría de peso:")
print(df_transformado["weight_category"].value_counts())

display(
    df_transformado[
        ["species", "body_mass_g", "weight_category"]
    ].head()
)

In [ ]:
df_biscoe = df_transformado[
    df_transformado["island"] == "Biscoe"
].copy()

print("Registros correspondientes a la isla Biscoe:")
print("Cantidad de filas:", len(df_biscoe))

display(df_biscoe.head())

In [ ]:
resumen_especies = (
    df_transformado
    .groupby("species", observed=True)["body_mass_g"]
    .mean()
    .round(2)
    .reset_index(name="peso_promedio_g")
)

display(resumen_especies)

In [ ]:
from src.resumidor_dataset import DatasetSummarizer

ruta_reporte = (
    ruta_proyecto / "reportes" / "resumen_dataset.txt"
)

resumidor = DatasetSummarizer(
    df,
    output_path=str(ruta_reporte)
)

resumidor.generar_resumen()

print("\n¿Se creó el reporte?", ruta_reporte.exists())

print("\nContenido del reporte:")
print(ruta_reporte.read_text(encoding="utf-8"))

In [ ]:
assert df.shape == (344, 8), "Las dimensiones originales no son las esperadas."

assert df_transformado[
    columnas_numericas + ["sex"]
].isnull().sum().sum() == 0, "Quedan valores faltantes en las columnas tratadas."

assert df_transformado["bill_ratio"].notna().all(), "Hay valores faltantes en bill_ratio."

assert df_transformado["weight_category"].notna().all(), "Hay registros sin categoría de peso."

assert ruta_reporte.exists(), "No se generó el archivo de reporte."

print("Todas las validaciones finalizaron correctamente.")